### Scenario 2: A cross-functional team with one data scientist working on an ML model
MLflow setup:

- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

mlflow server --backend-store-uri sqlite:///backend.db

In [4]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [5]:
print(f"tracking uri : '{mlflow.get_tracking_uri()}'")

tracking uri : 'http://127.0.0.1:5000'


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1772718780453, experiment_id='0', last_update_time=1772718780453, lifecycle_stage='active', name='Default', tags={}>]

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():
    X,y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state":42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X,y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/03/05 13:59:16 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/03/05 13:59:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/05 13:59:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 'mlflow-artifacts:/1/62c749fd1e634a94945e28509ea089e2/artifacts'
🏃 View run handsome-cub-281 at: http://127.0.0.1:5000/#/experiments/1/runs/62c749fd1e634a94945e28509ea089e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [8]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1772719156658, experiment_id='1', last_update_time=1772719156658, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1772718780453, experiment_id='0', last_update_time=1772718780453, lifecycle_stage='active', name='Default', tags={}>]

In [9]:
from mlflow.tracking import MlflowClient

client = MlflowClient("http://127.0.0.1:5000")

In [10]:
client.search_registered_models()

[]

In [ ]:
#run_id = client.list_run_infos(experiment_id='1')[0].run_id
runs = client.search_runs(experiment_ids=['1'])
run_id = runs[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name="iris-classifier"
)

Registered model 'iris-classifier' already exists. Creating a new version of this model...


MlflowException: Unable to find a logged_model with artifact_path models/ under run 62c749fd1e634a94945e28509ea089e2